***Import and load structure***

In [1]:
from Bio.PDB import PDBParser
from pathlib import Path

pdb_path = Path("../data/raw/1LYZ.pdb")

parser = PDBParser(QUIET=True)
structure = parser.get_structure("1LYZ", pdb_path)

print("Structure loaded:", structure)

Structure loaded: <Structure id=1LYZ>


***count models***

In [2]:
models = list(structure.get_models())
print("Number of models:", len(models))

Number of models: 1


***count chains***

In [3]:
model = models[0]
chains = list(model.get_chains())

print("Chains:")
for chain in chains:
    print(chain.id)

Chains:
A


***count residues***

In [4]:
from Bio.PDB.Polypeptide import is_aa

residue_count = 0

for residue in model.get_residues():
    if is_aa(residue, standard=True):
        residue_count += 1

print("Number of amino acid residues:", residue_count)

Number of amino acid residues: 129


***count atoms***

In [5]:
atom_count = sum(1 for _ in model.get_atoms())
print("Number of atoms:", atom_count)

Number of atoms: 1102


## Structure Summary (1LYZ)

- Models: 1
- Chains: 1 (Chain A)
- Residues: 129 amino acids
- Atoms: 1102

### Interpretation
This confirms that lysozyme is a small, single-chain protein, making it suitable for structure-based protein engineering and computational design workflows.

***Extract sequence***

In [6]:
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.IUPACData import protein_letters_3to1

model = structure[0]

sequence = []

for residue in model.get_residues():
    if is_aa(residue, standard=True):
        resname = residue.get_resname().capitalize()

        try:
            sequence.append(protein_letters_3to1[resname])
        except KeyError:
            sequence.append("X")  # unknown residue

sequence_str = "".join(sequence)

print("Sequence length:", len(sequence_str))
print("Sequence (first 50 aa):", sequence_str[:50])

Sequence length: 129
Sequence (first 50 aa): KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGS


***Add full sequence print- for visualization***

In [7]:
print(sequence_str)

KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL


***save sequence to file***

In [8]:
from pathlib import Path

output_path = Path("../data/processed/1LYZ_sequence.txt")

with open(output_path, "w") as f:
    f.write(sequence_str)

print("Saved sequence to:", output_path)

Saved sequence to: ../data/processed/1LYZ_sequence.txt


***verify file***

In [9]:
cat ../data/processed/1LYZ_sequence.txt

KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL

## Sequence–Structure Relationship

The sequence was extracted directly from the structure file rather than using a reference database.

### Why this matters
- ensures consistency with the actual structure used
- accounts for missing or unresolved residues
- avoids mismatch between sequence and structure

### Implication for protein engineering
All future mutations and designs will be based on this sequence, ensuring compatibility with structural constraints.

***creating a structured, reusable data table from Biopython objects***

In [10]:
import pandas as pd
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.IUPACData import protein_letters_3to1

model = structure[0]

residue_rows = []

for chain in model.get_chains():
    for residue in chain.get_residues():
        if not is_aa(residue, standard=True):
            continue

        hetflag, resseq, icode = residue.id
        resname_3 = residue.get_resname().capitalize()

        try:
            resname_1 = protein_letters_3to1[resname_3]
        except KeyError:
            resname_1 = "X"

        residue_rows.append({
            "chain_id": chain.id,
            "residue_number": resseq,
            "insertion_code": icode.strip() if isinstance(icode, str) else "",
            "residue_name_3": resname_3,
            "residue_name_1": resname_1,
        })

residue_df = pd.DataFrame(residue_rows)
residue_df.head()

,chain_id,residue_number,insertion_code,residue_name_3,residue_name_1
0,A,1,,Lys,K
1,A,2,,Val,V
2,A,3,,Phe,F
3,A,4,,Gly,G
4,A,5,,Arg,R


***check the total number of residues***

In [13]:
print("Total residues in residue table:", len(residue_df))

Total residues in residue table: 129


***quick summary cell***

In [14]:
print(residue_df["chain_id"].value_counts())
print()
print(residue_df["residue_name_1"].value_counts().head(10))

chain_id
A    129
Name: count, dtype: int64

residue_name_1
N    14
G    12
A    12
R    11
S    10
C     8
L     8
D     7
T     7
K     6
Name: count, dtype: int64


***Save the residue table***

In [15]:
from pathlib import Path

output_path = Path("../data/processed/residue_table.csv")
residue_df.to_csv(output_path, index=False)

print("Saved residue table to:", output_path)

Saved residue table to: ../data/processed/residue_table.csv


## Residue-Level Representation

This table converts the protein structure into a residue-level dataset.

### Why this matters
- protein engineering decisions are made at the residue level
- residues can now be filtered, counted, annotated, and scored
- this table will serve as the base for later structural and design features

### Current limitation
This table currently contains identity and numbering only. Structural context such as secondary structure, burial, or coordinates will be added later.